# Этап 2: рыночные признаки до `decision_at`

Цель — проверить, улучшают ли минутные market features предсказание значимости `|abnormal return 4h| ≥ 0,5%`, не используя ни одной цены, появившейся после решения.

## План и проверки

Сравниваем legacy, pre-publication, минимальный reaction core, расширенные price/microstructure-наборы и RuBERT ablations. Evaluation неизменен: 480 событий, семь месячных folds, два месяца validation, embargo 72 часа, purge по `event_group_id`. Свеча считается известной только после её полного окончания: `begin_at + 1 minute <= cutoff`. Primary metric — PR-AUC, неопределённость — cluster bootstrap по группе события.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
experiment_dir = cwd if (cwd / 'run_experiment.py').exists() else cwd / 'experiments' / 'market_stage2'
repo_root = experiment_dir.parents[1]
sys.path.insert(0, str(repo_root))
from experiments.market_stage2.run_experiment import run_experiment, sha256_file
from experiments.paths import dataset_path_from_environment, stage_artifact_directory

dataset_path = dataset_path_from_environment()
artifact_dir = stage_artifact_directory('market_stage2')
cache_path = artifact_dir / 'cache' / 'minute_candles.csv.gz'
print(dataset_path)

In [ ]:
# По умолчанию читаем уже проверенные артефакты. Для полного offline-пересчёта: RUN_STAGE2=1.
if os.environ.get('RUN_STAGE2') == '1':
    result = run_experiment(dataset_path, artifact_dir, offline=True)
else:
    result = json.loads((artifact_dir / 'metrics.json').read_text())
print(result['experiment'], result['created_at'])

In [ ]:
market = pd.read_csv(artifact_dir / 'market_features.csv')
material = pd.read_csv(artifact_dir / 'materiality_predictions.csv')
direction = pd.read_csv(artifact_dir / 'direction_predictions.csv')
stage1 = json.loads((stage_artifact_directory('finbert_stage1') / 'metrics.json').read_text())

assert result['dataset']['rows'] == 1238
assert len(material) == 480 and material.id.nunique() == 480
assert len(direction) == 480 and direction.id.nunique() == 480
assert material.fold.nunique() == 7
assert material.market_data_available.sum() == 472
assert result['market_cache']['failures'] == []
assert result['market_cache']['data_sha256'] == sha256_file(cache_path)
assert result['materiality']['legacy']['pr_auc'] == stage1['materiality']['baseline']['pr_auc']
assert result['materiality']['legacy']['roc_auc'] == stage1['materiality']['baseline']['roc_auc']
print('QA passed: cache hash, 480 paired rows, 7 folds, 98.33% market coverage, exact stage-1 legacy reproduction.')

## Значимость

In [ ]:
rows = []
for name, metrics in result['materiality'].items():
    rows.append({'feature set': name, 'PR-AUC': metrics['pr_auc'], 'ROC-AUC': metrics['roc_auc'], 'Brier': metrics['brier'], 'balanced accuracy': metrics['balanced_accuracy']})
display(pd.DataFrame(rows).set_index('feature set').round(4))
print('reaction_core vs legacy:')
print(json.dumps(result['materiality_paired_delta_vs_legacy']['reaction_core'], ensure_ascii=False, indent=2))
print('RuBERT vs reaction_core:')
print(json.dumps(result['finbert_delta_vs_reaction_core'], ensure_ascii=False, indent=2))

In [ ]:
coverage = pd.DataFrame(result['feature_coverage']).T
display(coverage.loc[['abnormal_reaction_0_5m_pct', 'stock_reaction_volume_ratio', 'stock_pre_return_60m_pct']].round(4))
display(pd.DataFrame(result['materiality_paired_delta_vs_legacy']['reaction_core']['pr_auc_by_fold']).round(4))

## Ручная временная сверка PHOR

In [ ]:
phor_id = 'sigex_bc61a274b843a0362a7e513b0b69bce5'
cols = ['stock_reaction_0_5m_pct', 'benchmark_reaction_0_5m_pct', 'abnormal_reaction_0_5m_pct', 'stock_publication_quote_age_minutes', 'stock_decision_quote_age_minutes', 'stock_reaction_candle_count']
display(market.loc[market.id == phor_id, cols].round(6))
print('Публикация 08:04:27Z; decision 08:09:27Z. Последние использованные свечи завершились в 08:04 и 08:09; entry 08:10 не используется.')

## Направление

In [ ]:
rows = []
for name, metrics in result['abnormal_direction'].items():
    rows.append({'feature set': name, 'hit rate': metrics['accuracy'], 'balanced accuracy': metrics['balanced_accuracy'], 'ROC-AUC': metrics['roc_auc']})
display(pd.DataFrame(rows).set_index('feature set').round(4))
display(pd.DataFrame(result['abnormal_direction_config6_population']['models']).T[['hit_rate', 'balanced_accuracy', 'hit_rate_delta_vs_config_6']].round(4))

In [ ]:
display(Image(filename=str(artifact_dir / 'materiality_feature_sets.png'))) 

## Вывод

Минимальный `reaction_core` — победитель development-теста: PR-AUC 0,6855 против 0,6419, положительный эффект во всех семи folds. RuBERT поверх core не улучшает PR-AUC. Направление остаётся нерешённым. Поскольку core-спецификация была выделена в ходе development-итерации, следующий шаг — заморозить её и проверить в shadow-режиме на новых, полностью нетронутых месяцах.